# AlphaZero — Connect 4 Training on Colab

Self-contained notebook to train an AlphaZero agent to play Connect 4 on a
Colab GPU with **checkpoint persistence to Google Drive**.

Checkpoints are written directly to Drive via a symlink, so training can be
resumed after a session disconnect without losing progress.

**Estimated wall-time:** ~3–5 hours on a free-tier T4 GPU.

**Config:** 6 residual blocks × 64 channels, 50 MCTS simulations, 30 iterations
(defined in `configs/connect4-quick.toml`).

---
**Repo:** https://github.com/venkata91/alphazero

## Step 1: Verify GPU

Before running this notebook make sure you have a GPU runtime attached:

- **Runtime → Change runtime type → T4 GPU** (free tier)
- Or select L4 / A100 if you have Colab Pro.

The cell below prints the GPU model and confirms PyTorch can see CUDA.
If it prints `CPU only`, stop and switch the runtime first.


In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print(
    "Device:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU only — change runtime to GPU",
)


## Step 2: Clone the Repo

Clones the AlphaZero repository from GitHub and changes into the project root.
If you re-run after a disconnect the clone will be skipped and we just `cd` in.


In [ ]:
import os

if not os.path.isdir("/content/alphazero"):
    !git clone https://github.com/venkata91/alphazero.git /content/alphazero
else:
    print("Repo already cloned — skipping.")

%cd /content/alphazero
!git log --oneline -3


## Step 3: Install Dependencies

Installs the `alphazero` package in editable mode along with all development
dependencies declared in `pyproject.toml`.

The `-q` flag suppresses most pip output; scroll up if you need to
inspect install logs.


In [ ]:
!pip install -q -e ".[dev]"
import alphazero
print("alphazero package version:", getattr(alphazero, "__version__", "(no __version__ attr)"))


## Step 4: Mount Google Drive for Checkpoint Persistence

Colab free-tier sessions can be terminated after ~12 hours of inactivity.
By writing checkpoints directly to Google Drive we ensure that completed
iterations are **never lost** and training can be resumed in a new session.

When the cell runs, Colab will open a browser popup asking you to sign in
to your Google account and grant Drive access.  Click through the prompts.

The checkpoint directory used is:
```
My Drive / alphazero / checkpoints / connect4
```


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_CKPT_DIR = "/content/drive/MyDrive/alphazero/checkpoints/connect4"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print("Drive checkpoint dir:", DRIVE_CKPT_DIR)
existing = sorted(os.listdir(DRIVE_CKPT_DIR))
print("Existing checkpoints:", existing or "(none)")


## Step 5: Set Up Checkpoint Symlink and Detect Resume Point

The Trainer writes checkpoints to `./checkpoints/` relative to the repo root.
We symlink that directory to the Drive folder so all writes go straight to
persistent storage.

We also scan the Drive directory for the latest `iter_NNNN.pt` checkpoint
so we can pass `--resume-from` to the Trainer in the next step.


In [ ]:
import os, glob, shutil

local_ckpt = "/content/alphazero/checkpoints"

# Remove a stale local (non-symlink) checkpoints dir if present
if os.path.exists(local_ckpt) and not os.path.islink(local_ckpt):
    shutil.rmtree(local_ckpt)
    print("Removed stale local checkpoints directory.")

# Create the symlink once
if not os.path.exists(local_ckpt):
    os.symlink(DRIVE_CKPT_DIR, local_ckpt)
    print(f"Symlinked {local_ckpt} -> {DRIVE_CKPT_DIR}")
else:
    print(f"Symlink already exists: {local_ckpt} -> {os.readlink(local_ckpt)}")

# Find the latest checkpoint to resume from
existing_ckpts = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/iter_*.pt"))
RESUME_FROM = existing_ckpts[-1] if existing_ckpts else None
print(f"Resume from: {RESUME_FROM or '(starting fresh)'}")


## Step 6: Train

Runs the AlphaZero trainer for Connect 4 using the config in
`configs/connect4-quick.toml`.  If a previous checkpoint was found on Drive the
`--resume-from` flag is passed so training continues from where it left off.

Training output is streamed live to this cell.  Checkpoints are saved after
each iteration to Drive automatically via the symlink.

> **Tip:** You can safely interrupt the cell (`Runtime → Interrupt execution`)
> at any time — the most recently completed checkpoint remains on Drive.

In [ ]:
import subprocess

cmd = [
    "python", "-m", "alphazero", "train",
    "--game", "connect4",
    "--config", "configs/connect4-quick.toml",
]

if RESUME_FROM is not None:
    cmd.extend(["--resume-from", RESUME_FROM])

print(f"Running: {' '.join(cmd)}\n")

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nExit code: {proc.returncode}")

## Step 7: Verify Final Checkpoint and Download

Lists all checkpoints saved to Drive and offers to download the latest one
to your local machine via the browser download dialog.

The file is typically ~2–10 MB depending on the network architecture size.


In [ ]:
import glob, os

ckpts = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/iter_*.pt"))
print(f"Total checkpoints on Drive: {len(ckpts)}")
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f"  {os.path.basename(c)}  ({size_mb:.1f} MB)")

if ckpts:
    latest = ckpts[-1]
    print(f"\nLatest checkpoint: {latest}")
    from google.colab import files
    files.download(latest)
else:
    print("No checkpoints found — training may not have completed successfully.")


## Step 8: Evaluate Against Minimax Depth-8 (Optional)

Runs 200 games of the trained policy network against a minimax-depth-8
opponent to measure playing strength.  This takes ~5–10 minutes.

A win rate above 80% against depth-8 indicates a strong agent.


In [ ]:
import glob

ckpts = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/iter_*.pt"))
final_ckpt = ckpts[-1] if ckpts else None

if final_ckpt:
    print(f"Evaluating checkpoint: {final_ckpt}")
    !python -m alphazero eval \
        --game connect4 \
        --checkpoint "{final_ckpt}" \
        --num-games 200
else:
    print("No checkpoint found to evaluate.")


## Interpreting the Output

- **Policy loss** measures how well the network predicts MCTS visit counts.
  It should decrease steadily over iterations.
- **Value loss** measures how well the network predicts game outcomes.
  Expect it to fluctuate early then stabilise.
- **Win rate vs minimax-depth-8**: a trained agent typically reaches >80%
  win rate after 40–50 iterations on the default Connect 4 config.

Checkpoints are stored at:
```
My Drive / alphazero / checkpoints / connect4 / iter_NNNN.pt
```

To continue training in a future session simply **Run all** again — the
notebook will automatically detect and resume from the latest checkpoint.

---

**Documentation & source:** https://github.com/venkata91/alphazero
